## Imports & API Setup

In [1]:
import re
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env file (create the .env file manually with your key)
load_dotenv(Path.cwd() / ".env", override=True)

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"

print("Imports done")
print("Groq client ready")
print(f"Model: {MODEL}")

Imports done
Groq client ready
Model: llama-3.3-70b-versatile


In [2]:
RUBRIC_CHUNKS = [

    #  Physics(Class 12) 
    {
        "id"       : "physics_class12",
        "subject"  : "Physics (Class 12)",
        "max_marks": 5,
        "keywords" : [
            "newton", "law", "motion", "force", "acceleration", "mass",
            "velocity", "momentum", "energy", "work", "power", "gravity",
            "electric", "magnetic", "current", "voltage", "resistance",
            "circuit", "wave", "frequency", "wavelength", "optics", "lens",
            "mirror", "refraction", "reflection", "thermodynamics", "heat",
            "kinetic", "potential", "nuclear", "atom", "electron", "proton",
            "formula", "derive", "derivation", "state", "explain", "define"
        ],
        "criteria" : [
            {"criterion": "Correct definition / statement of the law or concept", "marks": 2},
            {"criterion": "Correct formula with proper symbols and SI units",      "marks": 1},
            {"criterion": "Explanation of each term in the formula",               "marks": 1},
            {"criterion": "Example or application (if asked)",                     "marks": 1},
        ]
    },

    # Mathematics(Class 12)
    {
        "id"       : "mathematics_class12",
        "subject"  : "Mathematics (Class 12)",
        "max_marks": 5,
        "keywords" : [
            "matrix", "determinant", "integral", "derivative", "differentiate",
            "integrate", "limit", "continuity", "function", "vector", "probability",
            "permutation", "combination", "binomial", "sequence", "series",
            "arithmetic", "geometric", "trigonometry", "sine", "cosine", "tangent",
            "solve", "proof", "prove", "calculate", "find", "equation", "inequality"
        ],
        "criteria" : [
            {"criterion": "Correct method / approach chosen",           "marks": 1},
            {"criterion": "All steps shown clearly with justification",  "marks": 2},
            {"criterion": "Correct intermediate calculations",           "marks": 1},
            {"criterion": "Correct final answer with units if required", "marks": 1},
        ]
    },

    #  English(Class 10) 
    {
        "id"       : "english_class10",
        "subject"  : "English (Class 10)",
        "max_marks": 5,
        "keywords" : [
            "poem", "poet", "story", "author", "character", "theme", "moral",
            "passage", "comprehension", "essay", "letter", "paragraph",
            "describe", "narrate", "summarize", "summary", "meaning", "word",
            "sentence", "grammar", "tense", "voice", "active", "passive",
            "metaphor", "simile", "figure", "speech", "tone", "mood", "setting"
        ],
        "criteria" : [
            {"criterion": "Correct interpretation and relevance to question", "marks": 2},
            {"criterion": "Coverage of all key points",                       "marks": 1},
            {"criterion": "Clarity and logical flow of explanation",          "marks": 1},
            {"criterion": "Language quality and grammar",                     "marks": 1},
        ]
    },

    # Chemistry(Class 12) 
    {
        "id"       : "chemistry_class12",
        "subject"  : "Chemistry (Class 12)",
        "max_marks": 5,
        "keywords" : [
            "reaction", "element", "compound", "molecule", "bond", "ionic",
            "covalent", "acid", "base", "ph", "oxidation", "reduction", "redox",
            "electrolysis", "catalyst", "organic", "carbon", "hydrocarbon",
            "alkane", "alkene", "alcohol", "ester", "polymer", "periodic",
            "valence", "electron", "orbital", "equilibrium", "rate", "kinetics"
        ],
        "criteria" : [
            {"criterion": "Correct chemical concept / definition stated",  "marks": 2},
            {"criterion": "Correct equation or formula (if applicable)",   "marks": 1},
            {"criterion": "Explanation of mechanism or process",           "marks": 1},
            {"criterion": "Example compound or real-world application",    "marks": 1},
        ]
    },

    # Biology(Class 12)
    {
        "id"       : "biology_class12",
        "subject"  : "Biology (Class 12)",
        "max_marks": 5,
        "keywords" : [
            "cell", "nucleus", "mitosis", "meiosis", "dna", "rna", "gene",
            "chromosome", "protein", "enzyme", "photosynthesis", "respiration",
            "digestion", "circulation", "heart", "blood", "neuron", "brain",
            "hormone", "evolution", "ecosystem", "food", "chain", "species",
            "reproduction", "heredity", "genetics", "mutation", "organism"
        ],
        "criteria" : [
            {"criterion": "Correct biological definition or concept",            "marks": 2},
            {"criterion": "Accurate description of process or structure",        "marks": 1},
            {"criterion": "Correct diagram description or labelling (if asked)", "marks": 1},
            {"criterion": "Real-life example or significance",                   "marks": 1},
        ]
    },

    #  History / Social Science (Class 10) 
    {
        "id"       : "history_class10",
        "subject"  : "History / Social Science (Class 10)",
        "max_marks": 5,
        "keywords" : [
            "war", "revolution", "independence", "colonialism", "empire",
            "treaty", "nationalism", "democracy", "government", "parliament",
            "constitution", "rights", "freedom", "movement", "leader",
            "gandhi", "nehru", "british", "french", "world", "century",
            "economy", "trade", "industry", "agriculture", "society", "culture"
        ],
        "criteria" : [
            {"criterion": "Correct historical facts and dates",      "marks": 2},
            {"criterion": "Cause and effect explained clearly",      "marks": 1},
            {"criterion": "Key personalities or events mentioned",   "marks": 1},
            {"criterion": "Conclusion or significance of the event", "marks": 1},
        ]
    },

    # MANDATORY Fallback Rubric 
    {
        "id"       : "fallback_generic",
        "subject"  : "General (Fallback)",
        "max_marks": 5,
        "keywords" : [],   # empty -> never matched during scoring
        "criteria" : [
            {"criterion": "Relevance — answer directly addresses the question", "marks": 1},
            {"criterion": "Coverage — all key points included",                 "marks": 2},
            {"criterion": "Clarity — explanation is clear and logical",         "marks": 1},
            {"criterion": "Language — grammatically correct and precise",       "marks": 1},
        ]
    },
]

print(f"{len(RUBRIC_CHUNKS)} rubric chunks loaded (including fallback)")

7 rubric chunks loaded (including fallback)


## Define Rubric Chunks

Each rubric chunk has:
- id - unique name
- subject - broad subject area
- keywords - used for retrieval matching
- criteria - how marks are awarded
- max_marks - total marks available

- The last chunk is the mandatory fallback used when no subject matches.

In [3]:
def retrieve_rubric(question: str) -> dict:
    """
    Given a question string, return the most relevant rubric chunk.
    Uses keyword overlap scoring; falls back to the generic rubric.
    """
    question_tokens = set(re.findall(r"\b\w+\b", question.lower()))
    best_rubric = None
    best_score  = 0

    for rubric in RUBRIC_CHUNKS:
        if not rubric["keywords"]:
            continue
        score = len(question_tokens & set(rubric["keywords"]))
        if score > best_score:
            best_score  = score
            best_rubric = rubric

    if best_rubric is None or best_score == 0:
        best_rubric = next(r for r in RUBRIC_CHUNKS if r["id"] == "fallback_generic")
        print("No specific rubric matched — using fallback rubric.")
    else:
        print(f"Rubric matched: [{best_rubric['subject']}] (keyword score = {best_score})")

    return best_rubric


def format_rubric_for_prompt(rubric: dict) -> str:
    """Format rubric criteria into a readable string for the prompt."""
    lines = [
        f"Subject  : {rubric['subject']}",
        f"Max Marks: {rubric['max_marks']}",
        "",
        "Marking Criteria:"
    ]
    for i, c in enumerate(rubric["criteria"], 1):
        lines.append(f"  {i}. {c['criterion']}  [{c['marks']} mark(s)]")
    return "\n".join(lines)


#  Quick test
test_q  = "Explain Newton's second law of motion and state its formula."
matched = retrieve_rubric(test_q)
print(f"   → Matched ID: {matched['id']}")
print()
print(format_rubric_for_prompt(matched))

Rubric matched: [Physics (Class 12)] (keyword score = 6)
   → Matched ID: physics_class12

Subject  : Physics (Class 12)
Max Marks: 5

Marking Criteria:
  1. Correct definition / statement of the law or concept  [2 mark(s)]
  2. Correct formula with proper symbols and SI units  [1 mark(s)]
  3. Explanation of each term in the formula  [1 mark(s)]
  4. Example or application (if asked)  [1 mark(s)]


## Rubric Retrieval(Keyword Matching)

Approach:  
- Tokenise the question into lowercase words.  
- Count how many keywords from each rubric appear in the question.  
- Return the rubric with the highest score; fall back to generic if score = 0.


In [4]:
import re

SYSTEM_PROMPT = """\
You are a strict but fair academic examiner.
You will receive a question, a student's answer, and optionally a marking rubric.
Your task: evaluate the student's answer and return ONLY a valid JSON object with
exactly these four keys:
  "marks_awarded"  : integer — marks the student earns
  "max_marks"      : integer — maximum possible marks
  "feedback"       : string  — 1-3 sentence actionable feedback for the student
  "justification"  : string  — brief reason for the marks given, referencing criteria
Output ONLY the JSON object. No markdown, no backticks, no explanation outside JSON.
"""


def evaluate_answer(question     : str,
                    student_answer: str,
                    rubric        : dict,
                    use_rubric    : bool = True) -> dict:
    """
    Call Groq LLM to evaluate the student answer.
    Returns a dict: marks_awarded, max_marks, feedback, justification.
    """
    if use_rubric:
        rubric_section = f"""
MARKING RUBRIC
--------------
{format_rubric_for_prompt(rubric)}
"""
    else:
        rubric_section = f"""
No rubric provided. Evaluate holistically out of {rubric['max_marks']} marks
based on accuracy, completeness, and clarity.
"""

    user_message = f"""\
QUESTION
--------
{question}

STUDENT ANSWER
--------------
{student_answer}

{rubric_section}
Now evaluate and return only the JSON object.
"""

    # ── Groq API call (client.chat.completions.create) ───────────────────────
    response = client.chat.completions.create(
        model       = MODEL,
        max_tokens  = 512,
        temperature = 0.2,           # low temperature for consistent grading
        messages    = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
    )

    raw = response.choices[0].message.content.strip()

    # Strip accidental markdown fences
    raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
    raw = re.sub(r"\n?```$",          "", raw)

    try:
        result = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Groq returned non-JSON output:\n{raw}\n\nError: {e}")

    return result


print("evaluate_answer() defined (Groq LLM)")

evaluate_answer() defined (Groq LLM)


## LLM Evaluation Function

Prompt design decisions:
- Role: strict but fair examiner.
- Output: only valid JSON - no markdown fences, no preamble.
- Keys: marks_awarded, max_marks, feedback, justification.
- use_rubric=False mode for the bonus comparison.

In [6]:
def display_result(rubric: dict, result: dict, label: str = "") -> None:
    print(f"\n{'═'*55}")
    print(f"  Result {label}")
    print(f"{'═'*55}")
    print(f"  Subject      : {rubric['subject']}")
    print(f"  Marks Awarded: {result['marks_awarded']} / {result['max_marks']}")
    print(f"  Feedback     : {result['feedback']}")
    print(f"  Justification: {result['justification']}")
    print(f"{'─'*55}")


def run_evaluator(question      : str,
                  student_answer: str,
                  bonus_compare : bool = False) -> None:
    print("\n" + "="*55)
    print("   MINI ANSWER EVALUATOR  (powered by Groq)")
    print("="*55)
    print(f"  Question : {question}")
    print(f"  Answer   : {student_answer[:80]}..." if len(student_answer) > 80
          else f"  Answer   : {student_answer}")

    # Step 1: Retrieve rubric
    print("\nStep 1 — Retrieving rubric...")
    rubric = retrieve_rubric(question)
    print()
    print(format_rubric_for_prompt(rubric))

    # Step 2: Evaluate WITH rubric
    print("\nStep 2 — Evaluating with rubric...")
    result_with = evaluate_answer(question, student_answer, rubric, use_rubric=True)
    display_result(rubric, result_with, label="(With Rubric)")

    # Step 3 (Bonus): Compare with and without rubric
    if bonus_compare:
        print("\nStep 3 — Evaluating WITHOUT rubric (bonus comparison)...")
        result_without = evaluate_answer(question, student_answer, rubric, use_rubric=False)
        display_result(rubric, result_without, label="(Without Rubric)")

        print("\nCOMPARISON SUMMARY")
        print(f"{'─'*55}")
        print(f"  With Rubric    : {result_with['marks_awarded']} / {result_with['max_marks']}")
        print(f"  Without Rubric : {result_without['marks_awarded']} / {result_without['max_marks']}")
        diff = result_with['marks_awarded'] - result_without['marks_awarded']
        if diff > 0:
            print(f"  → Rubric awarded {diff} more mark(s) — more structured evaluation")
        elif diff < 0:
            print(f"  → Without rubric awarded {abs(diff)} more mark(s) — more lenient")
        else:
            print(f"  → Both methods gave the same marks")
        print(f"{'─'*55}")


print("run_evaluator() defined")

run_evaluator() defined


## Display and Evaluate the Helper

In [7]:
def display_result(rubric: dict, result: dict, label: str = "") -> None:
    print(f"\n{'═'*55}")
    print(f"  Result {label}")
    print(f"{'═'*55}")
    print(f"  Subject      : {rubric['subject']}")
    print(f"  Marks Awarded: {result['marks_awarded']} / {result['max_marks']}")
    print(f"  Feedback     : {result['feedback']}")
    print(f"  Justification: {result['justification']}")
    print(f"{'─'*55}")


def run_evaluator(question      : str,
                  student_answer: str,
                  bonus_compare : bool = False) -> None:
    print("\n" + "="*55)
    print("  🎓 MINI ANSWER EVALUATOR")
    print("="*55)
    print(f"  Question : {question}")
    print(f"  Answer   : {student_answer[:80]}..." if len(student_answer) > 80
          else f"  Answer   : {student_answer}")

    # Step 1: Retrieve rubric
    print("\nStep 1 — Retrieving rubric...")
    rubric = retrieve_rubric(question)
    print()
    print(format_rubric_for_prompt(rubric))

    # Step 2: Evaluate WITH rubric
    print("\nStep 2 — Evaluating with rubric...")
    result_with = evaluate_answer(question, student_answer, rubric, use_rubric=True)
    display_result(rubric, result_with, label="(With Rubric)")

    # Step 3 (Bonus): Compare with and without rubric
    if bonus_compare:
        print("\nStep 3 — Evaluating WITHOUT rubric (bonus comparison)...")
        result_without = evaluate_answer(question, student_answer, rubric, use_rubric=False)
        display_result(rubric, result_without, label="(Without Rubric)")

        print("\n📊 COMPARISON SUMMARY")
        print(f"{'─'*55}")
        print(f"  With Rubric    : {result_with['marks_awarded']} / {result_with['max_marks']}")
        print(f"  Without Rubric : {result_without['marks_awarded']} / {result_without['max_marks']}")
        diff = result_with['marks_awarded'] - result_without['marks_awarded']
        if diff > 0:
            print(f"  → Rubric awarded {diff} more mark(s) — more structured evaluation")
        elif diff < 0:
            print(f"  → Without rubric awarded {abs(diff)} more mark(s) — more lenient")
        else:
            print(f"  → Both methods gave the same marks")
        print(f"{'─'*55}")


print("✅ run_evaluator() defined")

✅ run_evaluator() defined


## Run Examples


## Example 1 - Physics(Good answer)

In [8]:
# Test 1: Physics - good answer
run_evaluator(
    question = "Explain Newton's second law of motion and state its formula.",
    student_answer = (
        "Newton's second law states that the force acting on an object is equal "
        "to the mass of the object multiplied by its acceleration. "
        "The formula is F = ma, where F is force in Newtons, m is mass in kilograms, "
        "and a is acceleration in m/s². "
        "This means a larger force produces a greater acceleration for the same mass."
    ),
    bonus_compare = True
)


  🎓 MINI ANSWER EVALUATOR
  Question : Explain Newton's second law of motion and state its formula.
  Answer   : Newton's second law states that the force acting on an object is equal to the ma...

Step 1 — Retrieving rubric...
Rubric matched: [Physics (Class 12)] (keyword score = 6)

Subject  : Physics (Class 12)
Max Marks: 5

Marking Criteria:
  1. Correct definition / statement of the law or concept  [2 mark(s)]
  2. Correct formula with proper symbols and SI units  [1 mark(s)]
  3. Explanation of each term in the formula  [1 mark(s)]
  4. Example or application (if asked)  [1 mark(s)]

Step 2 — Evaluating with rubric...

═══════════════════════════════════════════════════════
  Result (With Rubric)
═══════════════════════════════════════════════════════
  Subject      : Physics (Class 12)
  Marks Awarded: 4 / 5
  Feedback     : The answer is mostly correct, but lacks explanation of each term in the formula, which would have provided a clearer understanding of the concept.
  Justif

## Example 2 : Physics(Weak answer, missing formula)

In [9]:
run_evaluator(
    question = "Explain Newton's second law of motion and state its formula.",
    student_answer = "Newton's second law is about force and mass.",
    bonus_compare = False
)


  🎓 MINI ANSWER EVALUATOR
  Question : Explain Newton's second law of motion and state its formula.
  Answer   : Newton's second law is about force and mass.

Step 1 — Retrieving rubric...
Rubric matched: [Physics (Class 12)] (keyword score = 6)

Subject  : Physics (Class 12)
Max Marks: 5

Marking Criteria:
  1. Correct definition / statement of the law or concept  [2 mark(s)]
  2. Correct formula with proper symbols and SI units  [1 mark(s)]
  3. Explanation of each term in the formula  [1 mark(s)]
  4. Example or application (if asked)  [1 mark(s)]

Step 2 — Evaluating with rubric...

═══════════════════════════════════════════════════════
  Result (With Rubric)
═══════════════════════════════════════════════════════
  Subject      : Physics (Class 12)
  Marks Awarded: 0 / 5
  Feedback     : Please provide a complete definition of Newton's second law and its formula to improve your answer. Make sure to include the relationship between force, mass, and acceleration.
  Justification: 

## Example 3: English

In [10]:
run_evaluator(
    question = "Describe the theme of the poem 'The Road Not Taken' by Robert Frost.",
    student_answer = (
        "The poem is about making choices in life. The poet stands at a fork in the road "
        "and must choose one path. This represents the decisions we make in life. "
        "The theme is about individuality — choosing your own path even if it is less travelled. "
        "Frost uses the road as a metaphor for life choices."
    ),
    bonus_compare = True
)


  🎓 MINI ANSWER EVALUATOR
  Question : Describe the theme of the poem 'The Road Not Taken' by Robert Frost.
  Answer   : The poem is about making choices in life. The poet stands at a fork in the road ...

Step 1 — Retrieving rubric...
Rubric matched: [English (Class 10)] (keyword score = 3)

Subject  : English (Class 10)
Max Marks: 5

Marking Criteria:
  1. Correct interpretation and relevance to question  [2 mark(s)]
  2. Coverage of all key points  [1 mark(s)]
  3. Clarity and logical flow of explanation  [1 mark(s)]
  4. Language quality and grammar  [1 mark(s)]

Step 2 — Evaluating with rubric...

═══════════════════════════════════════════════════════
  Result (With Rubric)
═══════════════════════════════════════════════════════
  Subject      : English (Class 10)
  Marks Awarded: 4 / 5
  Feedback     : The student provides a clear and relevant interpretation of the poem's theme, but could further explore the poet's use of the road as a metaphor and its implications on individua

## Example 4 : Fallback(Unrecognised subject)

In [13]:
run_evaluator(
    question = "What are the main features of cloud computing?",
    student_answer = (
        "Cloud computing provides on-demand access to computing resources like servers, "
        "storage, and applications over the internet. Key features include scalability, "
        "flexibility, cost efficiency, and high availability. "
        "Examples include AWS, Google Cloud, and Microsoft Azure."
    ),
    bonus_compare = True
)


  🎓 MINI ANSWER EVALUATOR
  Question : What are the main features of cloud computing?
  Answer   : Cloud computing provides on-demand access to computing resources like servers, s...

Step 1 — Retrieving rubric...
No specific rubric matched — using fallback rubric.

Subject  : General (Fallback)
Max Marks: 5

Marking Criteria:
  1. Relevance — answer directly addresses the question  [1 mark(s)]
  2. Coverage — all key points included  [2 mark(s)]
  3. Clarity — explanation is clear and logical  [1 mark(s)]
  4. Language — grammatically correct and precise  [1 mark(s)]

Step 2 — Evaluating with rubric...

═══════════════════════════════════════════════════════
  Result (With Rubric)
═══════════════════════════════════════════════════════
  Subject      : General (Fallback)
  Marks Awarded: 4 / 5
  Feedback     : The answer is clear and concise, but could benefit from a more detailed explanation of the key features, such as how scalability and flexibility are achieved in cloud computing

## Interactive Mode(Enter Your Own Question)


In [15]:
print("🎓 Mini Answer Evaluator — Local Interactive Mode  (powered by Groq)")
print("-" * 60)

question       = input("📋 Enter the question       : ")
student_answer = input("✏️  Enter the student answer : ")
bonus          = input("📊 Compare with/without rubric? (y/n): ").strip().lower() == "y"

run_evaluator(question=question, student_answer=student_answer, bonus_compare=bonus)

🎓 Mini Answer Evaluator — Local Interactive Mode  (powered by Groq)
------------------------------------------------------------


📋 Enter the question       :  What is Force
✏️  Enter the student answer :  Force is mass multiplied by acceleration
📊 Compare with/without rubric? (y/n):  y



  🎓 MINI ANSWER EVALUATOR
  Question : What is Force
  Answer   : Force is mass multiplied by acceleration

Step 1 — Retrieving rubric...
Rubric matched: [Physics (Class 12)] (keyword score = 1)

Subject  : Physics (Class 12)
Max Marks: 5

Marking Criteria:
  1. Correct definition / statement of the law or concept  [2 mark(s)]
  2. Correct formula with proper symbols and SI units  [1 mark(s)]
  3. Explanation of each term in the formula  [1 mark(s)]
  4. Example or application (if asked)  [1 mark(s)]

Step 2 — Evaluating with rubric...

═══════════════════════════════════════════════════════
  Result (With Rubric)
═══════════════════════════════════════════════════════
  Subject      : Physics (Class 12)
  Marks Awarded: 3 / 5
  Feedback     : The student provided a correct definition and formula, but failed to explain the terms in the formula and did not provide an example or application.
  Justification: The student met criteria 1 and 2, but not criteria 3 and 4, hence awarded 3 out

## Gradio Interface

In [16]:
import gradio as gr


def gradio_evaluate(question: str, student_answer: str, compare_mode: bool) -> tuple:
    """
    Gradio-compatible wrapper around evaluate_answer().
    Returns (rubric_info, with_rubric_output, without_rubric_output, comparison_output)
    """
    if not question.strip():
        return "⚠️ Please enter a question.", "", "", ""
    if not student_answer.strip():
        return "⚠️ Please enter a student answer.", "", "", ""

    # Step 1: Retrieve rubric 
    rubric = retrieve_rubric(question)
    rubric_info = (
        f"**Subject:** {rubric['subject']}\n\n"
        f"**Max Marks:** {rubric['max_marks']}\n\n"
        f"**Criteria:**\n"
        + "\n".join(
            f"- {c['criterion']}  `[{c['marks']} mark(s)]`"
            for c in rubric['criteria']
        )
    )

    # Step 2: Evaluate WITH rubric
    try:
        result_with = evaluate_answer(question, student_answer, rubric, use_rubric=True)
    except Exception as e:
        return rubric_info, f"❌ Error: {e}", "", ""

    score_bar   = "🟩" * result_with['marks_awarded'] + "⬜" * (result_with['max_marks'] - result_with['marks_awarded'])
    with_output = (
        f"### {score_bar}  {result_with['marks_awarded']} / {result_with['max_marks']} marks\n\n"
        f"**📝 Feedback:**  {result_with['feedback']}\n\n"
        f"**🔍 Justification:**  {result_with['justification']}"
    )

    # Step 3 (optional): Evaluate WITHOUT rubric
    without_output = ""
    comparison_output = ""

    if compare_mode:
        try:
            result_without = evaluate_answer(question, student_answer, rubric, use_rubric=False)
        except Exception as e:
            without_output = f"❌ Error: {e}"
        else:
            score_bar2      = "🟩" * result_without['marks_awarded'] + "⬜" * (result_without['max_marks'] - result_without['marks_awarded'])
            without_output  = (
                f"### {score_bar2}  {result_without['marks_awarded']} / {result_without['max_marks']} marks\n\n"
                f"**📝 Feedback:**  {result_without['feedback']}\n\n"
                f"**🔍 Justification:**  {result_without['justification']}"
            )

            diff = result_with['marks_awarded'] - result_without['marks_awarded']
            if diff > 0:
                verdict = f"📊 Rubric awarded **{diff} more mark(s)** — more structured evaluation."
            elif diff < 0:
                verdict = f"📊 Without rubric awarded **{abs(diff)} more mark(s)** — more lenient."
            else:
                verdict = "📊 Both methods gave the **same marks**."

            comparison_output = (
                f"| Mode | Marks |\n"
                f"|------|-------|\n"
                f"| ✅ With Rubric    | {result_with['marks_awarded']} / {result_with['max_marks']} |\n"
                f"| ❌ Without Rubric | {result_without['marks_awarded']} / {result_without['max_marks']} |\n\n"
                + verdict
            )

    return rubric_info, with_output, without_output, comparison_output


# Example questions for the UI 
EXAMPLES = [
    [
        "Explain Newton's second law of motion and state its formula.",
        "Newton's second law states that force equals mass times acceleration (F = ma). "
        "F is in Newtons, m in kg, and a in m/s². A larger force produces greater acceleration.",
        True,
    ],
    [
        "Describe the theme of the poem 'The Road Not Taken' by Robert Frost.",
        "The poem is about choices in life. Frost uses a fork in the road as a metaphor. "
        "The theme is individuality — choosing your own path even if less travelled.",
        False,
    ],
    [
        "What are the main features of cloud computing?",
        "Cloud computing provides on-demand access to servers, storage, and apps over the internet. "
        "Key features: scalability, flexibility, cost efficiency. Examples: AWS, Azure, GCP.",
        True,
    ],
]


#  Build Gradio UI 
with gr.Blocks(
    title="🎓 Mini Answer Evaluator",
    theme=gr.themes.Soft(primary_hue="indigo"),
) as demo:

    gr.Markdown(
        """# 🎓 Mini Answer Evaluator
**Rubric-based student answer evaluation — powered by Groq LLM**

Enter a question and a student's answer. The system will auto-select the best rubric and score the answer."""
    )

    with gr.Row():
        with gr.Column(scale=1):
            question_box = gr.Textbox(
                label="📋 Question",
                placeholder="e.g. Explain Newton's second law of motion...",
                lines=3,
            )
            answer_box = gr.Textbox(
                label="✏️ Student Answer",
                placeholder="Type the student's answer here...",
                lines=6,
            )
            compare_toggle = gr.Checkbox(
                label="📊 Compare with / without rubric",
                value=False,
            )
            evaluate_btn = gr.Button("🚀 Evaluate", variant="primary", size="lg")

        with gr.Column(scale=1):
            rubric_display = gr.Markdown(label="📖 Matched Rubric")

    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### ✅ Result — With Rubric")
            with_rubric_out = gr.Markdown()
        with gr.Column(visible=True) as without_col:
            gr.Markdown("### ❌ Result — Without Rubric")
            without_rubric_out = gr.Markdown()

    comparison_out = gr.Markdown(label="📊 Comparison")

    gr.Examples(
        examples=EXAMPLES,
        inputs=[question_box, answer_box, compare_toggle],
        label="💡 Try an example",
    )

    evaluate_btn.click(
        fn=gradio_evaluate,
        inputs=[question_box, answer_box, compare_toggle],
        outputs=[rubric_display, with_rubric_out, without_rubric_out, comparison_out],
    )


demo.launch(share=True)   # set share=True to get a public Gradio link
print("✅ Gradio app launched — open the link above in your browser")

C:\Users\abhis\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
C:\Users\abhis\AppData\Local\Temp\ipykernel_4196\3083649923.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://553f1508bdf4bd02fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Gradio app launched — open the link above in your browser
Rubric matched: [Physics (Class 12)] (keyword score = 3)
Rubric matched: [Physics (Class 12)] (keyword score = 3)
